In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [2]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [3]:
retriever = vectorstore.as_retriever()

In [4]:
from langchain.tools import tool

@tool
def search_documents(query: str) -> str:
    """2026 년 테크노빌드 주식회사(TechnoBuild) 임직원 
통합 가이드북입니다.
    """
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [5]:
system_prompt = """당신은 테크노빌드 주식회사 가이드북 정보를 친절하게 제공하는 어시스턴트입니다.

1. 정보가 필요할 경우 반드시 검색 도구(retriever_tool)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 억지로 꾸며내지 말고 모른다고 답변하세요.
"""

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents],
    system_prompt=system_prompt
)

In [ ]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [8]:
from langchain.messages import SystemMessage, HumanMessage

res = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

In [9]:
print(res["messages"][-1].content[0]["text"])

테크노빌드 주식회사에서는 직무 관련 국가 기술 자격을 취득할 경우, **1회성 축하금**과 **매월 지급되는 자격 수당**을 받으실 수 있습니다. 자격 등급에 따른 상세 지급 금액은 다음과 같습니다.

### 1. 자격 등급별 지급 금액
| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |
| :--- | :--- | :--- | :--- |
| **기술사 / 기능장** | **200만 원** | **30만 원** | 금속재료, 용접, 기계가공 등 |
| **기사** | **50만 원** | **10만 원** | 일반기계, 전기, 산업안전 등 |
| **산업기사** | **30만 원** | **5만 원** | 기계설계, 위험물 등 |
| **기능사** | **10만 원** | **3만 원** | 선반, 밀링, 특수용접 등 |

### 2. 지급 조건 및 방법
*   **지급 조건:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정됩니다. (상위 등급 취득 시 수당 갱신)
*   **축하금:** 취득 횟수에 제한 없이 지급되며, 자격증 사본 제출 후 2주 이내에 별도로 입금됩니다.
*   **자격 수당:** HR팀에 자격증 사본을 제출한 익월 급여부터 반영됩니다.

자격증을 취득하셨다면 사본을 준비하여 HR팀에 제출하시기 바랍니다.


---

In [17]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents],
    system_prompt=system_prompt,
)

In [20]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [21]:
from langchain.messages import SystemMessage, HumanMessage

res = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

In [22]:
res

{'messages': [HumanMessage(content='자격증 비용은 얼마를 받을 수 있어?', additional_kwargs={}, response_metadata={}, id='7fd5e4d9-752c-4f6d-91f7-cc4e28e2b4ae'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_documents', 'arguments': '{"query": "\\uc790\\uaca9\\uc99d \\uc9c0\\uc6d0 \\ube44\\uc6a9 \\ubc0f \\uc218\\ub2f9 \\uc548\\ub0b4"}'}, '__gemini_function_call_thought_signatures__': {'006a2b6f-70b1-4315-b7b5-41aafebe37f4': 'EtYDCtMDAb4+9vvkxr9StXmIX9WbXeWd+7E3zqslNtBjTOEXi4HufG7JkbMs254bWiRr0D9RRZ+jONrnQn0gHNRoRg86WdAE2M/+2JBqpjsXFfWdxxZrI4gfIh91xYAu7j6zFarw/M9Aj0JVV7GpeehdBHeQ7aFMSEviZDKiwWmFJEyuggryrgVUn8HEx4nWeKLq8Qtu335/jJ57g+qsa9VHKMudEyovNZDOqlry2i+34Bbw24oTwflH3w3weI5icO2yqckKEwe3Ez4YnMPRQwseV3M+YmepE3o2HksEvkmAY2kGerUMTQNiLDRRBA3tA1eXwrv+GoI6g/lX3Q7YdunrPWsa6XvPS8NUmW7uDwAgKhz09V8/Z2VAnaZFxnCRAR3+nNsL1/8iuYSmfzQcojFq/faEvZWoTktc7qCcXvBl+3Y+Q/NGR0vg+zc1rXpZcbOd7d80y+as4lhkstbnmOip7OiQnKisOxbVCehNQLd6GjgZLSORAvFhbqLeKHlkCkn+CyDTmsr7V0rl4L2A08wMno0IBtfZivPMYL9vJEy

In [23]:
print(res["messages"][-1].content[0]["text"])

테크노빌드 주식회사에서는 직무 관련 국가 기술 자격 취득 시 **1회성 축하금**과 **매월 지급되는 자격 수당**을 받으실 수 있습니다. 자격 등급별 상세 금액은 다음과 같습니다.

### 1. 자격 등급별 지원 금액
| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |
| :--- | :--- | :--- | :--- |
| **기술사/기능장** | 200만 원 | 30만 원 | 금속재료, 용접, 기계가공 등 |
| **기사** | 50만 원 | 10만 원 | 일반기계, 전기, 산업안전 등 |
| **산업기사** | 30만 원 | 5만 원 | 기계설계, 위험물 등 |
| **기능사** | 10만 원 | 3만 원 | 선반, 밀링, 특수용접 등 |

### 2. 지급 조건 및 방법
*   **지급 조건:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정됩니다 (상위 등급 취득 시 갱신). 다만, **축하금은 횟수 제한 없이** 지급됩니다.
*   **신청 방법:** 자격증 사본을 HR팀에 제출하시면 됩니다.
*   **지급 시기:**
    *   **자격 수당:** 제출한 익월 급여부터 급여 명세서의 '자격 수당' 항목에 반영됩니다.
    *   **축하금:** 제출 후 2주 이내에 별도로 입금됩니다.


In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import SystemMessage, HumanMessage

# 딕셔너리에서 마지막 AI 메시지만 뽑아서 반환하는 단계
extract_ai_msg = lambda x: x["messages"][-1]

# 체인 구성: 에이전트 -> 메시지 추출 -> 문자열 변환
chain = agent | extract_ai_msg | StrOutputParser()

res = chain.invoke({
    "messages": [HumanMessage(content=query)]
})

In [27]:
res

'테크노빌드 주식회사에서는 직무 관련 국가 기술 자격 취득 시, 등급에 따라 **1회성 축하금**과 **매월 지급되는 자격 수당**을 받으실 수 있습니다. 상세 금액은 다음과 같습니다.\n\n### **자격 등급별 지급 금액**\n| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |\n| :--- | :--- | :--- | :--- |\n| **기술사 / 기능장** | 200만 원 | 30만 원 | 금속재료, 용접, 기계가공 등 |\n| **기사** | 50만 원 | 10만 원 | 일반기계, 전기, 산업안전 등 |\n| **산업기사** | 30만 원 | 5만 원 | 기계설계, 위험물 등 |\n| **기능사** | 10만 원 | 3만 원 | 선반, 밀링, 특수용접 등 |\n\n### **지급 조건 및 방법**\n*   **자격 수당:** 동일 등급 내에서는 1개의 자격증만 인정되며, 상위 등급 취득 시 수당이 갱신됩니다. 자격증 사본을 HR팀에 제출한 익월 급여부터 반영됩니다.\n*   **축하금:** 취득 횟수에 제한이 없으며, 자격증 제출 후 2주 이내에 별도로 입금됩니다.\n*   **기타:** 해당 업무에 종사할 경우 지급되는 것을 원칙으로 합니다.'